# TER Picopatt - AlphaEarth

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

from picopatt.io import load_all
import picopatt as fc

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.metrics import pairwise_distances

import ee

# Auth/Init
ee.Authenticate()
ee.Initialize(project="ee-acombesaguera")

# Tests
print("EE ping:", ee.Number(1).getInfo())

IC_AEF = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")
print("AlphaEarth IC size:", IC_AEF.size().getInfo())

img2024 = IC_AEF.filterDate("2024-01-01", "2025-01-01").first()
print("Bands example:", img2024.bandNames().getInfo()[:5])

In [ ]:
DATA_NOZERO = Path("data/processed/picopatt/clean_nozeros")
bd = load_all(DATA_NOZERO, False)

# Utilisation de lon_ontrack/lat_ontrack
required_geo = ["lon_ontrack", "lat_ontrack"]
missing_geo = [c for c in required_geo if c not in bd.columns]
if missing_geo:
    raise ValueError("Colonnes GNSS ontrack manquantes: " + ", ".join(missing_geo))

bd = bd.copy()
bd["lon_std"] = pd.to_numeric(bd["lon_ontrack"], errors="coerce")
bd["lat_std"] = pd.to_numeric(bd["lat_ontrack"], errors="coerce")

# Filtre coordonnées valides
bd = bd.dropna(subset=["lon_std", "lat_std"])
bd = bd[(bd["lon_std"].between(-180, 180)) & (bd["lat_std"].between(-90, 90))]

# UID stable
bd = bd.reset_index(drop=True)
bd["uid"] = bd.index.astype(int)

print("bd ready:", bd.shape)
print("Geo example:", bd[["uid","lon_std","lat_std"]].head())

In [ ]:
cols = ["uid", "track_id", "section_id", "lon_std", "lat_std"]
if "point_id" in bd.columns:
    cols.insert(3, "point_id")

pts = bd[cols].copy()
pts = pts.rename(columns={"lon_std":"longitude", "lat_std":"latitude"})

AED = Path("data/processed/alphaearth/alphaearth_data")
fc.create_folder(AED)

out_csv = AED / "picopatt_points.csv"
pts.to_csv(out_csv, index=False)
print("CSV points écrit:", out_csv, "n =", len(pts))

Avant de faire la suite du notebook, le csv ``picopatt_points.csv`` doit etre ajouté à earth engine via le bouton Assets puis New.

Ou simplement récupérer les exports AlphaEarth depuis le Drive du TER puis les placer dans ``data/external/alphaearth/picopatt_alphaearth``.

In [ ]:
BANDS = [f"A{i:02d}" for i in range(64)]

EXPORT_DIR = Path("data/external/alphaearth/picopatt_alphaearth")
files = sorted(EXPORT_DIR.glob("PICOPATT_AEF_points_2024_uid_*.csv"))
if not files:
    raise FileNotFoundError("Aucun export AlphaEarth trouvé. Vérifie EXPORT_DIR et les noms de fichiers.")

aef_pts = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

# Types + contrôles
aef_pts["uid"] = pd.to_numeric(aef_pts["uid"], errors="coerce").astype("Int64")

print("aef_pts shape:", aef_pts.shape)
print("uids uniques:", aef_pts["uid"].nunique(), "min:", aef_pts["uid"].min(), "max:", aef_pts["uid"].max())
print("missing A00:", aef_pts["A00"].isna().sum())

# Vérif bandes
missing_bands = [b for b in BANDS if b not in aef_pts.columns]
if missing_bands:
    raise ValueError("Bandes manquantes dans les exports: " + ", ".join(missing_bands[:10]))

In [ ]:
if "uid" not in bd.columns:
    bd = bd.reset_index(drop=True)
    bd["uid"] = bd.index.astype(int)

bd_enriched = bd.merge(aef_pts[["uid"] + BANDS], on="uid", how="left")

print("bd_enriched shape:", bd_enriched.shape)
print("Nb lignes sans embedding (A00 NaN):", bd_enriched["A00"].isna().sum())

In [ ]:
CANDIDATES = [
    "tair_thermohygro","tair_tc1","tair_tc2","tair_anemo",
    "rh_thermohygro","ws","wdir",
    "sw_up","sw_front","sw_right",
    "lw_up","lw_down","lw_front","lw_back","lw_left","lw_right",
    "tmrt","pet"
]

METEO = [c for c in CANDIDATES if c in bd_enriched.columns]

# wdir -> sin/cos
if "wdir" in METEO:
    theta = np.deg2rad(pd.to_numeric(bd_enriched["wdir"], errors="coerce") % 360)
    bd_enriched["wdir_sin"] = np.sin(theta)
    bd_enriched["wdir_cos"] = np.cos(theta)
    METEO = [c for c in METEO if c != "wdir"] + ["wdir_sin", "wdir_cos"]

CLUST = Path("prediction/figures/alphaearth/clusters")
fc.create_folder(CLUST)

AE = Path("data/processed/alphaearth/alphaearth_data")
fc.create_folder(AE)

In [ ]:
mask_ok = bd_enriched[BANDS].notna().all(axis=1)
n_drop = int((~mask_ok).sum())
if n_drop > 0:
    print("Lignes exclues PCA(64) (bandes manquantes):", n_drop)

X = bd_enriched.loc[mask_ok, BANDS].to_numpy()

pca64 = PCA(n_components=64, random_state=0)
pca64.fit(X)

explained = pca64.explained_variance_ratio_             
cum_explained = np.cumsum(explained)                     

df_pca_var = pd.DataFrame({
    "PC": np.arange(1, 65),
    "variance_expliquee": explained,
    "variance_expliquee_pct": explained * 100,
    "variance_cumulee": cum_explained,
    "variance_cumulee_pct": cum_explained * 100,
})

display(df_pca_var.head(64))

for thr in [0.5, 0.6, 0.7, 0.80, 0.90, 0.95]:
    n_comp = int(np.searchsorted(cum_explained, thr) + 1)
    print(f"Composantes nécessaires pour {int(thr*100)}% :", n_comp)

plt.figure(figsize=(14, 5))
x = np.arange(1, 65)

plt.bar(x, explained * 100, alpha=0.8, label="Variance expliquée (%)")
plt.plot(x, cum_explained * 100, marker="o", markersize=3, linewidth=1.5, label="Variance cumulée (%)")

plt.xticks(np.arange(1, 65, 2)) 
plt.xlabel("Composante principale (PC)")
plt.ylabel("Variance expliquée (%)")
plt.title("PCA AlphaEarth (A00..A63) – Variance expliquée (64 composantes)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

out_fig = CLUST / "alphaearth_pca64_explained_variance.png"
plt.savefig(out_fig, dpi=150)
plt.show()
print("Figure enregistrée:", out_fig)

out_csv = AE / "alphaearth_pca64_variance_table.csv"
df_pca_var.to_csv(out_csv, index=False)
print("Tableau variance exporté:", out_csv)

L’ACP explique 0,903 (90,3 %) de la variance totale avec 10 composantes. Cela indique que les 64 bandes contiennent une forte redondance, et qu’un vecteur réduit (de 10 composantes) conserve l’essentiel de l’information utile.

Avec 15 composantes, on obtient 95% de l'information. On gagne 5% d'information avec 5 composantes. 

In [ ]:
BANDS = [f"A{i:02d}" for i in range(64)]

AE = Path("data/processed/alphaearth/alphaearth_data")
fc.create_folder(AE)

if "uid" not in bd_enriched.columns:
    bd_enriched = bd_enriched.reset_index(drop=True)
    bd_enriched["uid"] = bd_enriched.index.astype(int)

X = bd_enriched[BANDS].to_numpy()

# PCA 10
pca10 = PCA(n_components=10, random_state=0)
Z10 = pca10.fit_transform(X)

PCA10_COLS = [f"aef_pca10_{i+1:02d}" for i in range(10)]
for i, c in enumerate(PCA10_COLS):
    bd_enriched[c] = Z10[:, i]

var10 = float(np.sum(pca10.explained_variance_ratio_))
print("Variance expliquée PCA10:", var10)

out_pca10 = bd_enriched[["uid"] + PCA10_COLS].copy()
pca10_path = AE / "alphaearth_pca10_points.csv"
out_pca10.to_csv(pca10_path, index=False)
print("Export PCA10:", pca10_path, ", shape:", out_pca10.shape)

# PCA 15
pca15 = PCA(n_components=15, random_state=0)
Z15 = pca15.fit_transform(X)

PCA15_COLS = [f"aef_pca15_{i+1:02d}" for i in range(15)]
for i, c in enumerate(PCA15_COLS):
    bd_enriched[c] = Z15[:, i]

var15 = float(np.sum(pca15.explained_variance_ratio_))
print("Variance expliquée PCA15:", var15)

out_pca15 = bd_enriched[["uid"] + PCA15_COLS].copy()
pca15_path = AE / "alphaearth_pca15_points.csv"
out_pca15.to_csv(pca15_path, index=False)
print("Export PCA15:", pca15_path, ", shape:", out_pca15.shape)

# PCA 32
pca32 = PCA(n_components=32, random_state=0)
Z32 = pca32.fit_transform(X)

PCA32_COLS = [f"aef_pca32_{i+1:02d}" for i in range(32)]
for i, c in enumerate(PCA32_COLS):
    bd_enriched[c] = Z32[:, i]

var32 = float(np.sum(pca32.explained_variance_ratio_))
print("Variance expliquée PCA32:", var32)

out_pca32 = bd_enriched[["uid"] + PCA32_COLS].copy()
pca32_path = AE / "alphaearth_pca32_points.csv"
out_pca32.to_csv(pca32_path, index=False)
print("Export PCA32:", pca32_path, ", shape:", out_pca32.shape)

# Export embeddings bruts A00..A63
out_raw = bd_enriched[["uid"] + BANDS].copy()
raw_path = AE / "alphaearth_A00_A63_points.csv"
out_raw.to_csv(raw_path, index=False)
print("Export A00..A63:", raw_path, ", shape:", out_raw.shape)

print(
    f"\nRésumé: \nPCA10={var10:.3f} \nPCA15={var15:.3f} \nPCA32={var32:.3f}"
    f"\n\nGain : \ngain15-10={var15-var10:.3f} \ngain32-15={var32-var15:.3f}"
)